### Kuliah #07 - Mengenal Operator: Pembuktian Konsep dan Implementasi Matriks

Notebook ini disusun sebagai pendamping modul kuliah **Serial Mekanika Kuantum Minimalis 2.0: Kuliah #07 - Mengenal Operator**. 

Secara struktur, notebook ini dibagi menjadi beberapa bagian utama sesuai dengan sub-bab di dalam diktat:
1. **Perilaku Dasar Operator**: Penulisan ulang persamaan, konsep transformasi keadaan, urutan operasi, dan deret fungsi operator.
2. **Contoh Operator: Rotasi Polarisasi**: Analisis operator rotasi $\hat{R}_p(	heta)$ dan pembuktian mengapa $\langle H|\hat{R}_p(45^\circ) \neq \langle +45|$.
3. **Operator Adjoint & Operator Uniter**: Sifat konjugat transpos, pembalikan urutan operasi, dan pelestarian magnitudo keadaan kuantum.
4. **Operator Proyeksi dan *Outer Product***: Konstruksi proyektor $\hat{P}_\psi = |\psi\rangle\langle\psi|$, sifat idempoten, dan hubungan kelengkapan (*completeness relation*).
5. **Representasi Matriks dari Operator**: Penurunan elemen matriks $O_{ij} = \langle i|\hat{O}|j\rangle$ dan konstruksi matriks dalam basis $HV$.
6. **Korespondensi antara Matriks Kuantum dan Klasik**: Perbedaan fisis antara matriks Jones klasik (seperti pelat setengah gelombang) dan operator rotasi kuantum, serta aktivitas optik.
7. **Operator Hermitian**: Sifat *self-adjoint*, nilai *eigen* riil, ortogonalitas vektor *eigen*, dan representasi spektral.
8. **Pembuktian Latihan Soal-Jawab (Soal 1 - 4)**: Penyelesaian lengkap secara analitik (aljabar Dirac & matriks) dan verifikasi komputasi Python.

Semua pembuktian matematis dijabarkan menggunakan aljabar matriks dan vektor kolom/baris yang diturunkan langsung dari persamaan Dirac. Kode Python menggunakan modul `numpy` dan `sympy` disediakan untuk memverifikasi setiap konsep secara numerik maupun simbolik.

In [13]:
import math
import numpy as np
import sympy as sp
from scipy.linalg import expm

np.set_printoptions(precision=4, suppress=True)
sp.init_printing()

def clean_array(A, tol=1e-12):
    """Membersihkan nilai elemen matriks/vektor yang mendekati nol akibat kesalahan pembulatan numerik (floating-point)."""
    A = np.array(A, dtype=complex)
    A[np.abs(A.real) < tol] = 1j * A[np.abs(A.real) < tol].imag
    A[np.abs(A.imag) < tol] = A[np.abs(A.imag) < tol].real
    if np.all(np.abs(A.imag) < tol):
        A = A.real
    return A

def print_matrix(name, M):
    M = clean_array(M)
    print(f"{name} =")
    print(M)
    print()

def print_ket(name, v):
    v = clean_array(v)
    print(f"|{name}> =")
    print(v)
    print()

# Fungsi dasar perkalian dalam (inner product) <bra|ket>
def inner_product(bra, ket):
    bra = np.array(bra, dtype=complex).flatten()
    ket = np.array(ket, dtype=complex).flatten()
    return np.vdot(bra, ket)

# Menghitung norma (panjang) vektor keadaan
def norm(v):
    return np.sqrt(np.abs(inner_product(v, v)))

# Operasi Adjoint (Conjugate Transpose)
def adjoint(M):
    return np.conjugate(np.transpose(M))

# Operasi Outer Product |ket><bra|
def outer_product(ket, bra):
    ket = np.array(ket, dtype=complex).reshape(-1, 1)
    bra = np.array(bra, dtype=complex).reshape(1, -1)
    return ket @ np.conjugate(bra)

# Definisi Vektor Keadaan Dasar dalam Basis HV
ket_H = np.array([[1], [0]], dtype=complex)
ket_V = np.array([[0], [1]], dtype=complex)

ket_plus45  = (1 / np.sqrt(2)) * np.array([[1], [1]], dtype=complex)
ket_minus45 = (1 / np.sqrt(2)) * np.array([[1], [-1]], dtype=complex)

ket_L = (1 / np.sqrt(2)) * np.array([[1], [1j]], dtype=complex)
ket_R = (1 / np.sqrt(2)) * np.array([[1], [-1j]], dtype=complex)

print("Setup selesai. Keadaan basis siap digunakan.")

Setup selesai. Keadaan basis siap digunakan.


### **1. Perilaku Dasar Operator**

#### 1.1 Penulisan Ulang Persamaan

Dalam mekanika kuantum, sebuah **operator** adalah objek matematis yang mentransformasikan suatu keadaan kuantum menjadi keadaan kuantum yang lain. Secara umum, operasi transformasi keadaan oleh operator $\hat{O}$ dituliskan sebagai:

$$\hat{O} |\psi\rangle = c |\psi'\rangle, \tag{1}$$

dengan $\hat{O}$ adalah operator yang mengubah keadaan kuantum $|\psi\rangle$ menjadi $|\psi'\rangle$, dan $c$ adalah konstanta bilangan kompleks.

Berdasarkan konvensi notasi Dirac, operator selalu ditempatkan **di sebelah luar** garis vertikal pada simbol keadaan (baik *bra* maupun *ket*). Oleh karena itu, penulisan yang tepat adalah $\hat{O}|\psi\rangle$ dan $\langle\psi|\hat{O}$, sedangkan penulisan $|\psi\rangle\hat{O}$ atau $\hat{O}\langle\psi|$ adalah salah secara konvensi.

Jika sebuah keadaan kuantum masukan $|\psi_i\rangle$ dioperasikan secara berurutan oleh serangkaian operator $\hat{O}_1, \hat{O}_2, \dots, \hat{O}_N$, maka transformasi terjadi secara bertahap dari kanan ke kiri:
- Setelah operasi pertama:

  $$|\psi_1\rangle = \hat{O}_1 |\psi_i\rangle. \tag{2}$$

- Setelah operasi kedua:

  $$|\psi_2\rangle = \hat{O}_2 |\psi_1\rangle = \hat{O}_2(\hat{O}_1 |\psi_i\rangle) = \hat{O}_2 \hat{O}_1 |\psi_i\rangle. \tag{3}$$

- Setelah $N$ operasi:

  $$|\psi_N\rangle = \hat{O}_N |\psi_{N-1}\rangle = \hat{O}_N \dots \hat{O}_2 \hat{O}_1 |\psi_i\rangle. \tag{4}$$

Sehingga operator efektif dari keseluruhan proses tersebut harus dituliskan dari **kanan ke kiri**:

$$\hat{O}_{eff} = \hat{O}_N \dots \hat{O}_2 \hat{O}_1. \tag{5}$$

Secara umum, operator tidak bersifat komutatif: $\hat{O}_2 \hat{O}_1 \neq \hat{O}_1 \hat{O}_2$. Walaupun demikian, sifat distributif tetap berlaku:

$$(\hat{O}_1 + \hat{O}_2) |\psi\rangle = \hat{O}_1 |\psi\rangle + \hat{O}_2 |\psi\rangle, \tag{6}$$

$$\hat{O}(|\psi_1\rangle + |\psi_2\rangle) = \hat{O}|\psi_1\rangle + \hat{O}|\psi_2\rangle. \tag{7}$$

Operasi perpangkatan pada operator didefinisikan melalui aplikasi berulang:

$$\hat{O}^2 = \hat{O}\hat{O}, \tag{8}$$

$$\hat{O}^n = \underbrace{\hat{O}\hat{O}\hat{O} \dots \hat{O}}_{n \text{ kali}}. \tag{9}$$

Fungsi dari suatu operator didefinisikan melalui deret Taylor/Maclaurin (deret pangkatnya). Sebagai contoh, fungsi eksponensial operator didefinisikan sebagai:

$$e^{\hat{O}} \equiv \sum_{n=0}^{\infty} \frac{1}{n!} \hat{O}^n. \tag{10}$$

---

## 1.2 Pembuktian Konsep dengan Operasi Matriks

Misalkan dalam basis ortonormal (seperti basis $HV$), operator $\hat{O}_1$ dan $\hat{O}_2$ direpresentasikan oleh matriks persegi $\mathbf{O}_1$ dan $\mathbf{O}_2$, serta vektor keadaan $|\psi\rangle$ direpresentasikan oleh vektor kolom $\mathbf{v}$.

### Bukti Urutan Operasi (Asosiatif Perkalian Matriks)
Aksi operator pertama $\hat{O}_1$ pada $|\psi_i\rangle$ menghasilkan vektor kolom baru:
$$\mathbf{v}_1 = \mathbf{O}_1 \mathbf{v}_i.$$
Ketika operator kedua $\hat{O}_2$ bekerja pada keadaan baru ini, kita memperoleh:
$$\mathbf{v}_2 = \mathbf{O}_2 \mathbf{v}_1 = \mathbf{O}_2 (\mathbf{O}_1 \mathbf{v}_i).$$
Karena perkalian matriks bersifat **asosiatif**, kita dapat mengelompokkan matriksnya:
$$\mathbf{v}_2 = (\mathbf{O}_2 \mathbf{O}_1) \mathbf{v}_i = \mathbf{O}_{eff} \mathbf{v}_i.$$
Terbukti bahwa matriks efektif adalah perkalian matriks dengan urutan dari kanan ke kiri: $\mathbf{O}_{eff} = \mathbf{O}_2 \mathbf{O}_1$.

### Bukti Ketidakkomutatifan ($\mathbf{O}_1 \mathbf{O}_2 \neq \mathbf{O}_2 \mathbf{O}_1$)
Dalam aljabar matriks, perkalian dua matriks umumnya tidak komutatif karena elemen matriks hasil kali pada baris $i$ dan kolom $j$ bergantung pada urutan produk titik baris dan kolom:
$$(\mathbf{O}_2 \mathbf{O}_1)_{ij} = \sum_k (O_2)_{ik} (O_1)_{kj} \quad \neq \quad (\mathbf{O}_1 \mathbf{O}_2)_{ij} = \sum_k (O_1)_{ik} (O_2)_{kj}.$$

In [14]:
# Demonstrasi Numerik Bagian 1: Perilaku Dasar Operator

# Kita definisikan dua operator matriks 2x2 sebagai contoh:
# O1 = Operator Proyeksi Horizontal P_H
O1 = np.array([[1, 0], 
               [0, 0]], dtype=complex)

# O2 = Operator Pelat Setengah Gelombang dengan sudut 22.5 derajat (merotasi H menjadi +45)
theta = np.deg2rad(22.5)
O2 = np.array([[np.cos(2*theta), np.sin(2*theta)],
               [np.sin(2*theta), -np.cos(2*theta)]], dtype=complex)

print_matrix("O1 (Proyeksi H)", O1)
print_matrix("O2 (Pelat Setengah Gelombang 22.5 deg)", O2)

# 1. Verifikasi Ketidakkomutatifan: O2 @ O1 != O1 @ O2
eff_21 = O2 @ O1  # O1 beroperasi dulu, lalu O2
eff_12 = O1 @ O2  # O2 beroperasi dulu, lalu O1

print_matrix("O_eff = O2 @ O1 (O1 beroperasi pertama)", eff_21)
print_matrix("O_eff = O1 @ O2 (O2 beroperasi pertama)", eff_12)
print("Apakah O2 @ O1 == O1 @ O2?", np.allclose(eff_21, eff_12))
print()

# 2. Verifikasi Sifat Distributif pada Vektor Keadaan masukan ket_plus45
lhs_dist = (O1 + O2) @ ket_plus45
rhs_dist = (O1 @ ket_plus45) + (O2 @ ket_plus45)
print("Verifikasi Sifat Distributif (O1 + O2)|psi> == O1|psi> + O2|psi>:")
print("Apakah sisi kiri sama dengan sisi kanan?", np.allclose(lhs_dist, rhs_dist))
print()

# 3. Verifikasi Deret Eksponensial e^O melalui ekspansi deret Taylor (persamaan 10)
# Kita hitung e^(X) di mana X = -i * pi/4 * sigma_z
X = -1j * (np.pi / 4) * np.array([[1, 0], [0, -1]], dtype=complex)

# Menggunakan fungsi eksponensial matriks tepat dari SciPy
exp_scipy = expm(X)

# Menggunakan deret Taylor sampai suku ke-20: sum(X^n / n!)
exp_taylor = np.zeros((2, 2), dtype=complex)
X_pow = np.eye(2, dtype=complex)
for n in range(25):
    exp_taylor = exp_taylor + (X_pow / float(math.factorial(n)))
    X_pow = X_pow @ X

print_matrix("e^X dari SciPy (expm)", exp_scipy)
print_matrix("e^X dari Deret Taylor (25 suku)", exp_taylor)
print("Apakah deret Taylor konvergen ke hasil eksponensial matriks?", np.allclose(exp_scipy, exp_taylor))

O1 (Proyeksi H) =
[[1. 0.]
 [0. 0.]]

O2 (Pelat Setengah Gelombang 22.5 deg) =
[[ 0.7071  0.7071]
 [ 0.7071 -0.7071]]

O_eff = O2 @ O1 (O1 beroperasi pertama) =
[[0.7071 0.    ]
 [0.7071 0.    ]]

O_eff = O1 @ O2 (O2 beroperasi pertama) =
[[0.7071 0.7071]
 [0.     0.    ]]

Apakah O2 @ O1 == O1 @ O2? False

Verifikasi Sifat Distributif (O1 + O2)|psi> == O1|psi> + O2|psi>:
Apakah sisi kiri sama dengan sisi kanan? True

e^X dari SciPy (expm) =
[[0.7071-0.7071j 0.    +0.j    ]
 [0.    +0.j     0.7071+0.7071j]]

e^X dari Deret Taylor (25 suku) =
[[0.7071-0.7071j 0.    +0.j    ]
 [0.    +0.j     0.7071+0.7071j]]

Apakah deret Taylor konvergen ke hasil eksponensial matriks? True


# 2. Contoh Operator: Rotasi Polarisasi

## 2.1 Penulisan Ulang Persamaan

Misalkan sebuah foton merambat pada sumbu-$z$ ($\vec{k} = k\vec{u}_z$) dan terpolarisasi linier pada keadaan $|H\rangle$. Kita ingin memutar polarisasinya mengelilingi sumbu perambatan sebesar sudut $45^\circ$ agar berakhir pada keadaan $|+45\rangle$. Operator rotasi polarisasi dinotasikan sebagai $\hat{R}_p(\theta)$, sehingga:

$$\hat{R}_p(45^\circ) |H\rangle = |+45\rangle. \tag{11}$$

Mengingat persamaan di atas, kita dapat bertanya apakah pernyataan berikut ini benar:

$$\langle H| \hat{R}_p(45^\circ) \stackrel{?}{=} \langle +45| \tag{12}$$

Untuk mengujinya, terapkan operator rotasi $\hat{R}_p(45^\circ)$ dua kali berturut-turut pada keadaan $|H\rangle$. Karena rotasi pertama menghasilkan $|+45\rangle$ dan rotasi kedua menambah sudut $45^\circ$ lagi, hasil total rotasi $90^\circ$ adalah polarisasi vertikal $|V\rangle$:

$$\hat{R}_p(45^\circ)\hat{R}_p(45^\circ) |H\rangle = \hat{R}_p(45^\circ) |+45\rangle = |V\rangle. \tag{13}$$

Jika kita ambil produk dalam (*inner product*) dengan $\langle H|$ dari sebelah kiri, kita memperoleh:

$$\langle H| \hat{R}_p(45^\circ)\hat{R}_p(45^\circ) |H\rangle = \langle H| \bigl[\hat{R}_p(45^\circ)\hat{R}_p(45^\circ) |H\rangle\bigr] = \langle H|V\rangle = 0. \tag{14}$$

Akan tetapi, apabila dugaan pada persamaan (12) itu benar, kita dapat mengelompokkan operasi pada persamaan sebelumnya sebagai perkalian antara *bra* $\langle H|\hat{R}_p(45^\circ)$ dan *ket* $\hat{R}_p(45^\circ)|H\rangle$:

$$\langle H| \hat{R}_p(45^\circ)\hat{R}_p(45^\circ) |H\rangle = \bigl[\langle H| \hat{R}_p(45^\circ)\bigr] \bigl[\hat{R}_p(45^\circ) |H\rangle\bigr] \stackrel{?}{=} \langle +45|+45\rangle = 1. \tag{15}$$

Terdapat kontradiksi yang nyata: persamaan (14) memberikan nilai **0**, sedangkan persamaan (15) memberikan nilai **1**. Hal ini membuktikan bahwa dugaan kita salah:

$$\langle H| \hat{R}_p(45^\circ) \neq \langle +45|. \tag{16}$$

---

## 2.2 Pembuktian Konsep dengan Operasi Matriks

Mari kita buktikan mengapa $\langle H| \hat{R}_p(45^\circ) \neq \langle +45|$ secara lugas menggunakan representasi matriks dan vektor.

Matriks operator rotasi polarisasi $\hat{R}_p(\theta)$ dalam basis $HV$ adalah:
$$\mathbf{R}_p(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}.$$
Untuk $\theta = 45^\circ$, karena $\cos(45^\circ) = \sin(45^\circ) = \frac{1}{\sqrt{2}}$, representasi matriksnya adalah:
$$\mathbf{R}_p(45^\circ) = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & -1 \\ 1 & 1 \end{pmatrix}.$$

Vektor *ket* $|H\rangle$ direpresentasikan oleh vektor kolom $\mathbf{v}_H = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$.
Aksi ke kanan pada $|H\rangle$:
$$\mathbf{R}_p(45^\circ) \mathbf{v}_H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & -1 \\ 1 & 1 \end{pmatrix} \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ 1 \end{pmatrix} = |+45\rangle. \quad (\text{Konsisten dengan Eq. 11})$$

Sekarang tinjau aksi ke kiri dari *bra* $\langle H|$, yang direpresentasikan oleh vektor baris $\mathbf{v}_H^T = \begin{pmatrix} 1 & 0 \end{pmatrix}$.
Jika kita mengalikan vektor baris $\langle H|$ secara langsung dengan matriks $\mathbf{R}_p(45^\circ)$ dari sebelah kanan:
$$\mathbf{v}_H^T \mathbf{R}_p(45^\circ) = \begin{pmatrix} 1 & 0 \end{pmatrix} \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & -1 \\ 1 & 1 \end{pmatrix} = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & -1 \end{pmatrix}.$$

Vektor baris $\frac{1}{\sqrt{2}} \begin{pmatrix} 1 & -1 \end{pmatrix}$ ini merupakan representasi *bra* dari polarisasi **$|-45\rangle$** ($\langle -45|$), bukan $\langle +45|$ (yang seharusnya $\frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \end{pmatrix}$).
Secara matematis:
$$\langle H| \hat{R}_p(45^\circ) = \langle -45| \neq \langle +45|.$$
Inilah akar penyebab kontradiksi pada persamaan (15).

In [15]:
# Demonstrasi Numerik Bagian 2: Mengapa <H| R_p(45) != <+45|

def R_p(theta_deg):
    theta = np.deg2rad(theta_deg)
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]], dtype=complex)

R_45 = R_p(45)
print_matrix("Matriks Rotasi R_p(45 deg)", R_45)

# 1. Verifikasi aksi ke kanan R_p(45)|H> == |+45>
ket_out = R_45 @ ket_H
print_ket("R_p(45) |H>", ket_out)
print("Apakah R_p(45)|H> sama dengan |+45>?", np.allclose(ket_out, ket_plus45))
print()

# 2. Verifikasi aksi ke kiri <H| R_p(45)
bra_H = ket_H.T.conj()  # Vektor baris [1, 0]
bra_out = bra_H @ R_45  # Hasil perkalian vektor baris dengan matriks

print("Vektor baris <H| R_p(45) =")
print(clean_array(bra_out))
print()

bra_plus45 = ket_plus45.T.conj()
bra_minus45 = ket_minus45.T.conj()

print("Vektor baris <+45| =", clean_array(bra_plus45))
print("Vektor baris <-45| =", clean_array(bra_minus45))
print()

print("Apakah <H| R_p(45) sama dengan <+45|?", np.allclose(bra_out, bra_plus45))
print("Apakah <H| R_p(45) sebenarnya sama dengan <-45|?", np.allclose(bra_out, bra_minus45))
print()

# 3. Pembuktian kontradiksi Eq. 14 vs Eq. 15
val_exact = bra_H @ R_45 @ R_45 @ ket_H
print(f"Perhitungan sebenarnya: <H| R_p(45) R_p(45) |H> = {val_exact[0,0]:.4f} (Benar = 0)")

val_wrong = (bra_plus45) @ (ket_plus45)
print(f"Perhitungan salah jika mengasumsikan <H|R_p(45) = <+45|: <+45|+45> = {val_wrong[0,0]:.4f} (Kontradiksi!)")

Matriks Rotasi R_p(45 deg) =
[[ 0.7071 -0.7071]
 [ 0.7071  0.7071]]

|R_p(45) |H>> =
[[0.7071]
 [0.7071]]

Apakah R_p(45)|H> sama dengan |+45>? True

Vektor baris <H| R_p(45) =
[[ 0.7071 -0.7071]]

Vektor baris <+45| = [[0.7071 0.7071]]
Vektor baris <-45| = [[ 0.7071 -0.7071]]

Apakah <H| R_p(45) sama dengan <+45|? False
Apakah <H| R_p(45) sebenarnya sama dengan <-45|? True

Perhitungan sebenarnya: <H| R_p(45) R_p(45) |H> = 0.0000+0.0000j (Benar = 0)
Perhitungan salah jika mengasumsikan <H|R_p(45) = <+45|: <+45|+45> = 1.0000+0.0000j (Kontradiksi!)


# 3. Operator Adjoint dan Operator Uniter

## 3.1 Penulisan Ulang Persamaan

Agar suatu operator memberikan transformasi yang ekuivalen ketika beroperasi ke arah kiri pada sebuah vektor *bra*, kita harus menggunakan **operator adjoint** (dinotasikan dengan simbol dagger $\dagger$):

$$\langle H| \hat{R}_p^\dagger(45^\circ) = \langle +45|. \tag{17}$$

Dengan menggunakan operator *adjoint*, perhitungan pada persamaan kontradiksi sebelumnya menjadi konsisten dan tepat:

$$\langle H| \hat{R}_p^\dagger(45^\circ) \hat{R}_p(45^\circ) |H\rangle = \bigl[\langle H| \hat{R}_p^\dagger(45^\circ)\bigr] \bigl[\hat{R}_p(45^\circ) |H\rangle\bigr] = \langle +45|+45\rangle = 1. \tag{18}$$

Secara umum, jika kita mengetahui transformasi pada *ket*: $\hat{O}|c_1\rangle = |c_2\rangle$, maka dengan mengambil *adjoint* dari kedua sisi, kita memperoleh hubungan pada *bra*: $\langle c_1| \hat{O}^\dagger = \langle c_2|$.
Sifat penting dari *adjoint* untuk suatu perkalian operator adalah **urutan operasinya menjadi terbalik**:

$$(\hat{O}_1 \hat{O}_2)^\dagger = \hat{O}_2^\dagger \hat{O}_1^\dagger. \tag{19}$$

### Operator Uniter
Perhatikan kembali persamaan (18). Dengan pengelompokan operasi secara berbeda pada bagian tengahnya:

$$\langle H| \hat{R}_p^\dagger(45^\circ) \hat{R}_p(45^\circ) |H\rangle = \langle H| \bigl[\hat{R}_p^\dagger(45^\circ) \hat{R}_p(45^\circ) |H\rangle\bigr] = \langle H|\psi\rangle = 1. \tag{20}$$

Agar $\langle H|\psi\rangle = 1$ terpenuhi untuk vektor ber-norma 1, haruslah disyaratkan bahwa $|\psi\rangle = |H\rangle$. Dengan demikian:

$$\hat{R}_p^\dagger(45^\circ) \hat{R}_p(45^\circ) |H\rangle = |H\rangle, \tag{21}$$

$$\hat{R}_p^\dagger(45^\circ) \hat{R}_p(45^\circ) = \hat{1}, \tag{22}$$

dengan $\hat{1}$ adalah operator identitas. Karena tidak ada keistimewaan khusus pada sudut $45^\circ$, secara umum berlaku untuk sembarang sudut $\theta$:

$$\hat{R}_p^\dagger(\theta) \hat{R}_p(\theta) = \hat{1}. \tag{23}$$

Operator rotasi merupakan salah satu contoh dari **operator uniter** ($\hat{U}$). Definisi matematis dari operator uniter adalah:

$$\hat{U}^\dagger \hat{U} = \hat{U} \hat{U}^\dagger = \hat{1}. \tag{24}$$

Keistimewaan utama operator uniter adalah senantiasa **menjaga normalisasi (magnitudo)** dari suatu keadaan kuantum:

$$\hat{U}|c_1\rangle = e^{i\phi} |c_2\rangle. \tag{25}$$

Invers dari operator $\hat{O}$ dinotasikan dengan $\hat{O}^{-1}$ dan didefinisikan oleh:

$$\hat{O}^{-1} \hat{O} = \hat{O} \hat{O}^{-1} = \hat{1}. \tag{26}$$

Khusus untuk operator uniter, berlaku $\hat{U}^{-1} = \hat{U}^\dagger$. Apabila kita memutar keadaan sebesar $45^\circ$, lalu diputar balik sebesar $-45^\circ$, kita kembali ke keadaan awal:

$$\hat{R}_p(-45^\circ) \hat{R}_p(45^\circ) |H\rangle = |H\rangle. \tag{27}$$

Membandingkan persamaan ini dengan persamaan (22), kita menyimpulkan bahwa $\hat{R}_p^\dagger(45^\circ) = \hat{R}_p(-45^\circ)$, atau secara umum:

$$\hat{R}_p^\dagger(\theta) = \hat{R}_p^{-1}(\theta) = \hat{R}_p(-\theta). \tag{28}$$

---

## 3.2 Pembuktian Konsep dengan Operasi Matriks

### Bukti Operasi Adjoint pada Matriks ($M^\dagger = (M^*)^T$)
Dalam representasi matriks, operasi *adjoint* $\dagger$ adalah **konjugat transpos** (*Hermitian conjugate*). Jika $|\psi\rangle \doteq \mathbf{v}$, maka *bra* $\langle\psi| \doteq \mathbf{v}^\dagger = (\mathbf{v}^*)^T$.
Misalkan $|\psi'\rangle = \hat{O}|\psi\rangle \doteq \mathbf{O}\mathbf{v}$. Jika kita mengambil transpos konjugat dari kedua sisi:
$$(\mathbf{O}\mathbf{v})^\dagger = ((\mathbf{O}\mathbf{v})^*)^T = (\mathbf{O}^* \mathbf{v}^*)^T = (\mathbf{v}^*)^T (\mathbf{O}^*)^T = \mathbf{v}^\dagger \mathbf{O}^\dagger.$$
Hal ini membuktikan aturan pembalikan urutan pada persamaan (19): $(\mathbf{A}\mathbf{B})^\dagger = \mathbf{B}^\dagger \mathbf{A}^\dagger$, dan menunjukkan bahwa *bra* $\langle\psi'|$ diperoleh dengan mengalikan $\langle\psi|$ dengan $\mathbf{O}^\dagger$ dari sebelah kanan.

### Bukti Pelestarian Norma oleh Operator Uniter
Misalkan $|\psi'\rangle = \hat{U}|\psi\rangle$. Kita hitung kuadrat norma dari $|\psi'\rangle$:
$$\langle \psi'|\psi'\rangle = (\hat{U}|\psi\rangle)^\dagger (\hat{U}|\psi\rangle) = \langle\psi| \hat{U}^\dagger \hat{U} |\psi\rangle.$$
Karena $\hat{U}$ uniter ($\hat{U}^\dagger \hat{U} = \hat{1}$), maka:
$$\langle \psi'|\psi'\rangle = \langle\psi| \hat{1} |\psi\rangle = \langle\psi|\psi\rangle.$$
Panjang vektor kuantum tidak berubah setelah ditransformasikan oleh operator uniter.

In [16]:
# Demonstrasi Numerik Bagian 3: Operator Adjoint dan Operator Uniter

R_45 = R_p(45)
R_45_adj = adjoint(R_45)

print_matrix("R_p(45)", R_45)
print_matrix("R_p(45)^dagger", R_45_adj)

# 1. Verifikasi Eq. 17: <H| R_p(45)^dagger == <+45|
bra_H = ket_H.T.conj()
bra_result = bra_H @ R_45_adj
print("Vektor baris <H| R_p(45)^dagger =", clean_array(bra_result))
print("Vektor baris <+45|              =", clean_array(ket_plus45.T.conj()))
print("Apakah <H| R_p(45)^dagger == <+45|?", np.allclose(bra_result, ket_plus45.T.conj()))
print()

# 2. Verifikasi sifat pembalikan urutan (AB)^dagger = B^dagger @ A^dagger
A = np.array([[1+1j, 2], [3-2j, 4j]], dtype=complex)
B = np.array([[0, 1j], [-1j, 2]], dtype=complex)
LHS = adjoint(A @ B)
RHS = adjoint(B) @ adjoint(A)
print("Verifikasi (AB)^dagger == B^dagger A^dagger:")
print("Apakah LHS == RHS?", np.allclose(LHS, RHS))
print()

# 3. Verifikasi Keuniteran R_p(theta): R^dagger @ R == I
theta_test = 37.5
R_test = R_p(theta_test)
unitarity_check = adjoint(R_test) @ R_test
print_matrix(f"R_p({theta_test} deg)^dagger @ R_p({theta_test} deg)", unitarity_check)
print("Apakah matriks identitas I?", np.allclose(unitarity_check, np.eye(2)))
print()

# 4. Verifikasi Pelestarian Norma untuk sembarang keadaan (misal keadaan eliptis sembarang)
ket_sembarang = (1/np.sqrt(3)) * ket_H + np.sqrt(2/3) * 1j * ket_V
ket_berputar = R_test @ ket_sembarang

norm_awal = norm(ket_sembarang)
norm_akhir = norm(ket_berputar)
print(f"Norma sebelum rotasi = {norm_awal:.6f}")
print(f"Norma setelah rotasi = {norm_akhir:.6f}")
print("Apakah norma terjaga persis sama?", np.isclose(norm_awal, norm_akhir))

R_p(45) =
[[ 0.7071 -0.7071]
 [ 0.7071  0.7071]]

R_p(45)^dagger =
[[ 0.7071  0.7071]
 [-0.7071  0.7071]]

Vektor baris <H| R_p(45)^dagger = [[0.7071 0.7071]]
Vektor baris <+45|              = [[0.7071 0.7071]]
Apakah <H| R_p(45)^dagger == <+45|? True

Verifikasi (AB)^dagger == B^dagger A^dagger:
Apakah LHS == RHS? True

R_p(37.5 deg)^dagger @ R_p(37.5 deg) =
[[1. 0.]
 [0. 1.]]

Apakah matriks identitas I? True

Norma sebelum rotasi = 1.000000
Norma setelah rotasi = 1.000000
Apakah norma terjaga persis sama? True


# 4. Operator Proyeksi dan *Outer Product*

## 4.1 Penulisan Ulang Persamaan

Sebuah polarisator horizontal $PA_{HV}$ menyiapkan foton dalam keadaan $|H\rangle$. Untuk keadaan sembarang $|\psi\rangle = c_H|H\rangle + c_V|V\rangle$, operator proyeksi $\hat{P}_H$ memproyeksikan $|\psi\rangle$ ke arah $|H\rangle$:

$$\hat{P}_H |\psi\rangle = c_H |H\rangle. \tag{29}$$

Dengan membedah koefisien $c_H = \langle H|\psi\rangle$, kita memperoleh bentuk perkalian luar (*outer product*):

$$\hat{P}_H |\psi\rangle = c_H |H\rangle = |H\rangle c_H = |H\rangle \langle H|\psi\rangle = (|H\rangle \langle H|) |\psi\rangle. \tag{30}$$

Sehingga operator proyeksi pada $|H\rangle$ didefinisikan oleh $\hat{P}_H = |H\rangle\langle H|$. Secara umum, operator proyeksi pada sembarang keadaan ter-normalisasi $|\psi\rangle$ diberikan oleh:

$$\hat{P}_\psi = |\psi\rangle \langle \psi|. \tag{31}$$

Berbeda dengan *inner product* $\langle\psi_1|\psi_2\rangle$ yang menghasilkan **bilangan skalar**, *outer product* $|\psi_1\rangle\langle\psi_2|$ menghasilkan sebuah **operator (matriks)**.

Dalam basis $HV$, sembarang keadaan $|\psi\rangle$ dapat diuraikan sebagai:

$$|\psi\rangle = c_H|H\rangle + c_V|V\rangle = |H\rangle\langle H|\psi\rangle + |V\rangle\langle V|\psi\rangle = (\hat{P}_H + \hat{P}_V)|\psi\rangle. \tag{32}$$

Karena $|\psi\rangle$ adalah sembarang keadaan, haruslah berlaku bahwa jumlah operator proyeksi pada basis ortonormal menghasilkan operator identitas:

$$\hat{P}_H + \hat{P}_V = \hat{1}. \tag{33}$$

Secara umum, jika himpunan keadaan $\{|j\rangle\}$ membentuk basis ortonormal yang lengkap, berlaku **hubungan kelengkapan (*completeness relation*)**:

$$\hat{1} = \sum_j \hat{P}_j = \sum_j |j\rangle\langle j|. \tag{34}$$

---

## 4.2 Pembuktian Konsep dengan Operasi Matriks

### Bukti Matriks *Outer Product* dan Kelengkapan Basis $HV$
Secara aljabar linier, *outer product* antara vektor kolom $\mathbf{v} = \begin{pmatrix} v_1 \\ v_2 \end{pmatrix}$ dan vektor baris $\mathbf{u}^\dagger = \begin{pmatrix} u_1^* & u_2^* \end{pmatrix}$ adalah matriks $2 \times 2$:
$$\mathbf{v} \mathbf{u}^\dagger = \begin{pmatrix} v_1 \\ v_2 \end{pmatrix} \begin{pmatrix} u_1^* & u_2^* \end{pmatrix} = \begin{pmatrix} v_1 u_1^* & v_1 u_2^* \\ v_2 u_1^* & v_2 u_2^* \end{pmatrix}.$$

Untuk basis $|H\rangle \doteq \begin{pmatrix} 1 \\ 0 \end{pmatrix}$ dan $|V\rangle \doteq \begin{pmatrix} 0 \\ 1 \end{pmatrix}$:
$$\mathbf{P}_H = |H\rangle\langle H| = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \begin{pmatrix} 1 & 0 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix},$$
$$\mathbf{P}_V = |V\rangle\langle V| = \begin{pmatrix} 0 \\ 1 \end{pmatrix} \begin{pmatrix} 0 & 1 \end{pmatrix} = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix}.$$
Jika kita jumlahkan kedua matriks proyektor tersebut:
$$\mathbf{P}_H + \mathbf{P}_V = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix} + \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = \mathbf{I}.$$
Terbukti secara matriks bahwa $\sum_j |j\rangle\langle j| = \hat{1}$.

### Bukti Sifat Idempoten Operator Proyeksi ($\hat{P}_\psi^2 = \hat{P}_\psi$)
Dengan menggunakan aturan aljabar Dirac untuk keadaan ter-normalisasi ($\langle\psi|\psi\rangle = 1$):
$$\hat{P}_\psi^2 = (|\psi\rangle\langle\psi|) (|\psi\rangle\langle\psi|) = |\psi\rangle (\langle\psi|\psi\rangle) \langle\psi| = |\psi\rangle (1) \langle\psi| = |\psi\rangle\langle\psi| = \hat{P}_\psi.$$
Memproyeksikan suatu keadaan ke ruang yang sama dua kali berturut-turut memberikan hasil yang sama persis dengan satu kali proyeksi.

In [17]:
# Demonstrasi Numerik Bagian 4: Operator Proyeksi dan Outer Product

# Konstruksi matriks proyektor dari outer_product
P_H = outer_product(ket_H, ket_H)
P_V = outer_product(ket_V, ket_V)
P_plus45 = outer_product(ket_plus45, ket_plus45)
P_minus45 = outer_product(ket_minus45, ket_minus45)
P_L = outer_product(ket_L, ket_L)
P_R = outer_product(ket_R, ket_R)

print_matrix("P_H = |H><H|", P_H)
print_matrix("P_V = |V><V|", P_V)
print_matrix("P_+45 = |+45><+45|", P_plus45)
print_matrix("P_L = |L><L|", P_L)

# 1. Verifikasi Sifat Idempoten P^2 == P
print("Verifikasi Sifat Idempoten P^2 == P:")
for name, P in [("P_H", P_H), ("P_V", P_V), ("P_+45", P_plus45), ("P_L", P_L)]:
    print(f"Apakah {name}^2 == {name}?", np.allclose(P @ P, P))
print()

# 2. Verifikasi Hubungan Kelengkapan (Completeness Relation) sum |j><j| == I
print("Verifikasi Hubungan Kelengkapan untuk berbagai basis ortonormal:")
print("Apakah P_H + P_V == I?", np.allclose(P_H + P_V, np.eye(2)))
print("Apakah P_+45 + P_-45 == I?", np.allclose(P_plus45 + P_minus45, np.eye(2)))
print("Apakah P_L + P_R == I?", np.allclose(P_L + P_R, np.eye(2)))
print()

# 3. Verifikasi aksi P_H pada keadaan |psi> = c_H|H> + c_V|V>
psi = 0.6 * ket_H + 0.8 * 1j * ket_V
hasil_proyeksi = P_H @ psi
ekspektasi = inner_product(ket_H, psi) * ket_H
print_ket("P_H |psi>", hasil_proyeksi)
print("Apakah P_H |psi> == <H|psi> |H>?", np.allclose(hasil_proyeksi, ekspektasi))

P_H = |H><H| =
[[1. 0.]
 [0. 0.]]

P_V = |V><V| =
[[0. 0.]
 [0. 1.]]

P_+45 = |+45><+45| =
[[0.5 0.5]
 [0.5 0.5]]

P_L = |L><L| =
[[ 0.5+0.j  -0. -0.5j]
 [ 0. +0.5j  0.5+0.j ]]

Verifikasi Sifat Idempoten P^2 == P:
Apakah P_H^2 == P_H? True
Apakah P_V^2 == P_V? True
Apakah P_+45^2 == P_+45? True
Apakah P_L^2 == P_L? True

Verifikasi Hubungan Kelengkapan untuk berbagai basis ortonormal:
Apakah P_H + P_V == I? True
Apakah P_+45 + P_-45 == I? True
Apakah P_L + P_R == I? True

|P_H |psi>> =
[[0.6]
 [0. ]]

Apakah P_H |psi> == <H|psi> |H>? True


# 5. Representasi Matriks dari Operator

## 5.1 Penulisan Ulang Persamaan

Ketika menyatakan keadaan sebagai vektor kolom, kita melabeli basis-basis dengan bilangan bulat:

$$|H\rangle = |1\rangle, \qquad |V\rangle = |2\rangle. \tag{35}$$

Sebuah keadaan umum dinyatakan sebagai kombinasi linier basis:

$$|\psi\rangle = c_1 |1\rangle + c_2 |2\rangle = \sum_j c_j |j\rangle, \tag{36}$$

dengan koefisien $c_j = \langle j|\psi\rangle$ (Persamaan 37). Representasi vektor kolom dari $|\psi\rangle$ adalah:

$$|\psi\rangle \doteq \begin{pmatrix} \langle 1|\psi\rangle \\ \langle 2|\psi\rangle \end{pmatrix}_{HV} = \begin{pmatrix} c_1 \\ c_2 \end{pmatrix}_{HV}. \tag{38}$$

Misalkan operator $\hat{O}$ beroperasi menghasilkan keadaan baru $|\psi'\rangle = \hat{O}|\psi\rangle$ (Persamaan 39), dengan vektor kolom $|\psi'\rangle \doteq \begin{pmatrix} c'_1 \\ c'_2 \end{pmatrix}_{HV}$ (Persamaan 40). Kita dapat menentukan elemen $c'_i$ dengan menyisipkan operator identitas dari hubungan kelengkapan:

$$c'_i = \langle i|\psi'\rangle = \langle i|\hat{O}|\psi\rangle = \langle i|\hat{O}\hat{1}|\psi\rangle, \tag{41}$$

$$c'_i = \langle i|\hat{O} \left(\sum_j |j\rangle\langle j|\right) |\psi\rangle = \sum_j \langle i|\hat{O}|j\rangle \langle j|\psi\rangle = \sum_j \langle i|\hat{O}|j\rangle c_j. \tag{42}$$

Kita definisikan elemen matriks pada baris $i$ dan kolom $j$ sebagai:

$$O_{ij} \equiv \langle i|\hat{O}|j\rangle. \tag{43}$$

Maka transformasi keadaan dituliskan dalam bentuk sumasi:

$$c'_i = \sum_j O_{ij} c_j. \tag{44}$$

Persamaan ini ekuivalen persis dengan perkalian matriks linier standar pada ruang 2 dimensi:

$$\begin{pmatrix} c'_1 \\ c'_2 \end{pmatrix}_{HV} = \begin{pmatrix} O_{11} & O_{12} \\ O_{21} & O_{22} \end{pmatrix}_{HV} \begin{pmatrix} c_1 \\ c_2 \end{pmatrix}_{HV}. \tag{45}$$

### Contoh 1: Representasi Matriks Operator Proyeksi $\hat{P}_H$
Berdasarkan definisi (Persamaan 46-48):

$$\hat{P}_H \doteq \begin{pmatrix} \langle H|\hat{P}_H|H\rangle & \langle H|\hat{P}_H|V\rangle \\ \langle V|\hat{P}_H|H\rangle & \langle V|\hat{P}_H|V\rangle \end{pmatrix}_{HV} = \begin{pmatrix} \langle H|H\rangle & 0 \\ \langle V|H\rangle & 0 \end{pmatrix}_{HV} = \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix}_{HV}.$$

### Contoh 2: Representasi Matriks Operator Rotasi Polarisasi $\hat{R}_p(\theta)$
Aksi operator rotasi pada masing-masing keadaan basis (Persamaan 49-50):

$$\hat{R}_p(\theta)|1\rangle = \hat{R}_p(\theta)|H\rangle = \cos\theta |H\rangle + \sin\theta |V\rangle, \tag{49}$$

$$\hat{R}_p(\theta)|2\rangle = \hat{R}_p(\theta)|V\rangle = -\sin\theta |H\rangle + \cos\theta |V\rangle. \tag{50}$$

Dengan menghitung produk dalam $\langle i|\hat{R}_p(\theta)|j\rangle$, diperoleh representasi matriksnya:

$$\hat{R}_p(\theta) \doteq \begin{pmatrix} \langle H| [\cos\theta|H\rangle + \sin\theta|V\rangle] & \langle H| [-\sin\theta|H\rangle + \cos\theta|V\rangle] \\ \langle V| [\cos\theta|H\rangle + \sin\theta|V\rangle] & \langle V| [-\sin\theta|H\rangle + \cos\theta|V\rangle] \end{pmatrix}_{HV} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}_{HV}. \tag{51}$$

---

## 5.2 Pembuktian Konsep dengan Operasi Matriks

Kunci utama dalam membangun representasi matriks dari sebuah operator adalah: **Kolom ke-$j$ dari matriks operator $\mathbf{O}$ adalah vektor koordinat dari hasil aksi operator tersebut pada basis ke-$j$**.
Secara eksplisit, jika $\hat{O}|1\rangle = a|1\rangle + b|2\rangle 	o \begin{pmatrix} a \\ b \end{pmatrix}$ dan $\hat{O}|2\rangle = c|1\rangle + d|2\rangle 	o \begin{pmatrix} c \\ d \end{pmatrix}$, maka matriksnya adalah $\begin{pmatrix} a & c \\ b & d \end{pmatrix}$.

In [18]:
# Demonstrasi Numerik Bagian 5: Membangun Representasi Matriks dari Fungsi Aksi Operator

def build_matrix_representation(action_func, basis_list):
    """Membangun matriks operator O dengan menghitung elemen O_{ij} = <basis_i | O | basis_j>."""
    dim = len(basis_list)
    M = np.zeros((dim, dim), dtype=complex)
    for i in range(dim):
        for j in range(dim):
            ket_j_transformed = action_func(basis_list[j])
            M[i, j] = inner_product(basis_list[i], ket_j_transformed)
    return clean_array(M)

# 1. Membangun matriks untuk P_H dari fungsi aksinya: action(psi) = <H|psi> |H>
def aksi_PH(psi):
    return inner_product(ket_H, psi) * ket_H

M_PH = build_matrix_representation(aksi_PH, [ket_H, ket_V])
print_matrix("Matriks P_H (hasil rekonstruksi dari elemen <i|O|j>)", M_PH)

# 2. Membangun matriks untuk R_p(theta) dari definisi aksi fisik Eq. 49 & 50
theta_deg = 30
theta_rad = np.deg2rad(theta_deg)

def aksi_rotasi(basis_ket):
    if np.allclose(basis_ket, ket_H):
        return np.cos(theta_rad)*ket_H + np.sin(theta_rad)*ket_V
    elif np.allclose(basis_ket, ket_V):
        return -np.sin(theta_rad)*ket_H + np.cos(theta_rad)*ket_V

M_rot = build_matrix_representation(aksi_rotasi, [ket_H, ket_V])
print_matrix(f"Matriks R_p({theta_deg} deg) (rekonstruksi dari aksi pada basis)", M_rot)
print("Apakah sama dengan fungsi R_p standar?", np.allclose(M_rot, R_p(theta_deg)))

Matriks P_H (hasil rekonstruksi dari elemen <i|O|j>) =
[[1. 0.]
 [0. 0.]]

Matriks R_p(30 deg) (rekonstruksi dari aksi pada basis) =
[[ 0.866 -0.5  ]
 [ 0.5    0.866]]

Apakah sama dengan fungsi R_p standar? True


# 6. Korespondensi antara Matriks Kuantum dan Klasik

## 6.1 Penulisan Ulang Persamaan

Pada kuliah tentang kalkulus Jones, kita mengetahui bahwa matriks Jones untuk polarisator horizontal dan pelat gelombang bersesuaian dengan representasi matriks operator kuantum dalam basis $HV$.

Namun terdapat perbedaan penting pada operator rotasi: **Tidak ada elemen pelat setengah gelombang tunggal yang bersesuaian langsung dengan operator rotasi $\hat{R}_p(\theta)$**.
Alasannya: Pelat setengah gelombang dengan sudut sumbu cepat $\alpha$ merotasi polarisasi linier masukan bersudut $\phi$ menjadi polarisasi bersudut $2\alpha - \phi$. Artinya, jumlah rotasinya bergantung pada sudut masukan $\phi$.

Sebaliknya, operator rotasi kuantum $\hat{R}_p(\theta)$ merotasi **sembarang** keadaan polarisasi linier $|\phi\rangle$ dengan sudut tambahan yang seragam sebesar $\theta$:

$$\hat{R}_p(\theta) |\phi\rangle = |\phi + \theta\rangle. \tag{53}$$

Implementasi fisis yang nyata dari operator $\hat{R}_p(\theta)$ dalam optik klasik maupun kuantum bukanlah pelat gelombang, melainkan material yang memiliki **aktivitas optik** (seperti kristal kuarsa atau larutan gula pasir). Material ini memutar bidang polarisasi cahaya sebesar sudut yang sebanding dengan ketebalan medium, tidak peduli berapa sudut polarisasi masukannya.

---

## 6.2 Pembuktian Konsep dengan Operasi Matriks

Mari kita buktikan secara aljabar linier bahwa $\mathbf{R}_p(\theta) \mathbf{v}_\phi = \mathbf{v}_{\phi+\theta}$.
Vektor keadaan terpolarisasi linier pada sudut $\phi$ adalah:
$$\mathbf{v}_\phi = \begin{pmatrix} \cos\phi \\ \sin\phi \end{pmatrix}.$$
Kalikan dengan matriks rotasi $\mathbf{R}_p(\theta)$:
$$\mathbf{R}_p(\theta) \mathbf{v}_\phi = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} \cos\phi \\ \sin\phi \end{pmatrix} = \begin{pmatrix} \cos\theta\cos\phi - \sin\theta\sin\phi \\ \sin\theta\cos\phi + \cos\theta\sin\phi \end{pmatrix}.$$
Dengan menggunakan identitas trigonometri penjumlahan sudut:
- $\cos\theta\cos\phi - \sin\theta\sin\phi = \cos(\phi + \theta)$
- $\sin\theta\cos\phi + \cos\theta\sin\phi = \sin(\phi + \theta)$

Sehingga diperoleh:
$$\mathbf{R}_p(\theta) \mathbf{v}_\phi = \begin{pmatrix} \cos(\phi + \theta) \\ \sin(\phi + \theta) \end{pmatrix} = |\phi + \theta\rangle.$$

In [19]:
# Demonstrasi Numerik & Simbolik Bagian 6: Rotasi Seragam R_p(theta)|phi> = |phi + theta>

# 1. Pembuktian Simbolik menggunakan SymPy
theta_sym, phi_sym = sp.symbols('theta phi', real=True)
R_sym = sp.Matrix([[sp.cos(theta_sym), -sp.sin(theta_sym)],
                   [sp.sin(theta_sym),  sp.cos(theta_sym)]])
ket_phi_sym = sp.Matrix([[sp.cos(phi_sym)],
                         [sp.sin(phi_sym)]])

ket_rot_sym = sp.simplify(R_sym * ket_phi_sym)
print("Hasil perkalian simbolik R_p(theta) * |phi>:")
sp.pprint(ket_rot_sym)
print()

# 2. Verifikasi Numerik untuk berbagai pasangan sudut (phi, theta)
sudut_tes = [(15, 30), (45, 45), (60, -15), (0, 90)]
print("Verifikasi Numerik R_p(theta)|phi> == |phi + theta>:")
for phi_val, theta_val in sudut_tes:
    ket_phi = np.array([[np.cos(np.deg2rad(phi_val))], [np.sin(np.deg2rad(phi_val))]])
    ket_expected = np.array([[np.cos(np.deg2rad(phi_val + theta_val))], [np.sin(np.deg2rad(phi_val + theta_val))]])
    ket_actual = R_p(theta_val) @ ket_phi
    cocok = np.allclose(ket_actual, ket_expected)
    print(f"  phi = {phi_val:>3} deg, theta = {theta_val:>3} deg -> Cocok? {cocok}")

Hasil perkalian simbolik R_p(theta) * |phi>:
⎡cos(φ + θ)⎤
⎢          ⎥
⎣sin(φ + θ)⎦

Verifikasi Numerik R_p(theta)|phi> == |phi + theta>:
  phi =  15 deg, theta =  30 deg -> Cocok? True
  phi =  45 deg, theta =  45 deg -> Cocok? True
  phi =  60 deg, theta = -15 deg -> Cocok? True
  phi =   0 deg, theta =  90 deg -> Cocok? True


# 7. Operator Hermitian

## 7.1 Penulisan Ulang Persamaan

Ingat kembali bahwa jika $\hat{O}|\psi_j\rangle = |\psi'_j\rangle$, maka elemen matriks dari operator adjoin $\hat{O}^\dagger$ adalah:

$$O_{ij}^\dagger = \langle \psi_i | \hat{O}^\dagger | \psi_j \rangle = \langle \psi'_i | \psi_j \rangle = \langle \psi_j | \psi'_i \rangle^* = \langle \psi_j | \hat{O} | \psi_i \rangle^* = O_{ji}^*. \tag{54}$$

Terdapat kelompok operator khusus yang bersifat *self-adjoint*, yaitu $\hat{O}^\dagger = \hat{O}$. Operator semacam ini disebut sebagai **operator Hermitian**. Elemen matriks dari operator Hermitian memenuhi relasi kesimetrian konjugat:

$$O_{ij} = O_{ji}^*. \tag{54b}$$

Persamaan masalah nilai *eigen* (*eigenvalue problem*) untuk operator $\hat{O}$ dituliskan sebagai:

$$\hat{O} |\lambda_i\rangle = \lambda_i |\lambda_i\rangle. \tag{55}$$

Operator Hermitian memiliki dua sifat fundamental dalam mekanika kuantum (akan dibuktikan secara lengkap pada Soal-Jawab):
1. **Nilai *eigen* ($\lambda_i$) selalu berupa bilangan riil**.
2. **Vektor *eigen* ($|\lambda_i\rangle$) yang bersesuaian dengan nilai *eigen* berbeda saling ortogonal**.

Karena vektor-vektor *eigen* dari operator Hermitian pada ruang Hilbert berdimensi berhingga membentuk himpunan ortonormal yang lengkap, kita dapat menuliskan hubungan kelengkapan dalam basis *eigen* tersebut:

$$\hat{1} = \sum_i |\lambda_i\rangle \langle \lambda_i|. \tag{56}$$

Dengan menyisipkan identitas ini, sembarang operator Hermitian $\hat{\mathcal{H}}$ dapat dinyatakan dalam bentuk **dekomposisi spektral**:

$$\hat{\mathcal{H}} = \hat{\mathcal{H}}\hat{1} = \sum_i \hat{\mathcal{H}} |\lambda_i\rangle \langle \lambda_i| = \sum_i \lambda_i |\lambda_i\rangle \langle \lambda_i|. \tag{57}$$

---

## 7.2 Pembuktian Konsep dengan Operasi Matriks

### Bukti Elemen Diagonal Matriks Hermitian Selalu Riil
Untuk elemen diagonal ($i = j$), relasi $O_{ij} = O_{ji}^*$ memberikan:
$$O_{ii} = O_{ii}^*.$$
Satu-satunya bilangan kompleks yang sama dengan konjugat kompleksnya sendiri adalah bilangan riil ($a + bi = a - bi \implies b = 0$).

### Bukti Dekomposisi Spektral Matriks ($\mathbf{H} = \mathbf{V} \mathbf{\Lambda} \mathbf{V}^\dagger$)
Misalkan matriks $\mathbf{V} = \begin{pmatrix} |\lambda_1\rangle & |\lambda_2\rangle \dots |\lambda_N\rangle \end{pmatrix}$ adalah matriks uniter yang kolom-kolomnya adalah vektor *eigen* ortonormal dari $\mathbf{H}$.
Maka perkalian $\sum_i \lambda_i |\lambda_i\rangle \langle \lambda_i|$ ekuivalen dengan:
$$\mathbf{V} \begin{pmatrix} \lambda_1 & & 0 \\ & \ddots & \\ 0 & & \lambda_N \end{pmatrix} \mathbf{V}^\dagger = \mathbf{H}.$$

In [20]:
# Demonstrasi Numerik Bagian 7: Operator Hermitian & Representasi Spektral

# Kita periksa apakah proyektor P_+45 dan operator rotasi R_p(45) bersifat Hermitian
print("Apakah P_+45 Hermitian (P == P^dagger)?", np.allclose(P_plus45, adjoint(P_plus45)))
print("Apakah R_p(45) Hermitian (R == R^dagger)?", np.allclose(R_45, adjoint(R_45)))
print()

# Kita ambil contoh operator Hermitian matriks Pauli-X (sigma_x) dan proyektor P_+45
H_mat = P_plus45
evals, evecs = np.linalg.eigh(H_mat)

print("Nilai eigen dari P_+45:", evals)
print("Apakah nilai eigen sepenuhnya riil?", np.allclose(evals.imag, 0))
print()

# Vektor eigen dari np.linalg.eigh tersimpan sebagai kolom-kolom matriks evecs
ket_lam0 = evecs[:, 0:1] # Vektor untuk nilai eigen 0
ket_lam1 = evecs[:, 1:2] # Vektor untuk nilai eigen 1

print_ket("Vektor eigen |lam_0> (untuk lam=0, yaitu |-45>)", ket_lam0)
print_ket("Vektor eigen |lam_1> (untuk lam=1, yaitu |+45>)", ket_lam1)

# Verifikasi ortogonalitas vektor eigen
ip_eigen = inner_product(ket_lam0, ket_lam1)
print(f"Inner product <lam_0 | lam_1> = {ip_eigen:.6f} (Ortogonal)")
print()

# Verifikasi Representasi Spektral Eq. 57: H == sum lam_i |lam_i><lam_i|
H_spektral = evals[0] * outer_product(ket_lam0, ket_lam0) + evals[1] * outer_product(ket_lam1, ket_lam1)
print_matrix("Rekonstruksi Spektral sum lam_i |lam_i><lam_i|", H_spektral)
print("Apakah sama persis dengan matriks asal P_+45?", np.allclose(H_mat, H_spektral))

Apakah P_+45 Hermitian (P == P^dagger)? True
Apakah R_p(45) Hermitian (R == R^dagger)? False

Nilai eigen dari P_+45: [0. 1.]
Apakah nilai eigen sepenuhnya riil? True

|Vektor eigen |lam_0> (untuk lam=0, yaitu |-45>)> =
[[-0.7071]
 [ 0.7071]]

|Vektor eigen |lam_1> (untuk lam=1, yaitu |+45>)> =
[[0.7071]
 [0.7071]]

Inner product <lam_0 | lam_1> = 0.000000+0.000000j (Ortogonal)

Rekonstruksi Spektral sum lam_i |lam_i><lam_i| =
[[0.5 0.5]
 [0.5 0.5]]

Apakah sama persis dengan matriks asal P_+45? True


# 8. Pembuktian Latihan Soal-Jawab (Soal 1 - 4)

## Soal 1
**Buktikan bahwa:**
**(i) operator uniter $\hat{U}$ tidak mengubah magnitudo dari suatu vektor keadaan $|\psi\rangle$,**
**(ii) kuadrat operator proyeksi pada $\psi$ adalah operator proyeksi itu sendiri ($\hat{P}_\psi^2 = \hat{P}_\psi$).**

### Pembuktian Analitik:
**(i)** Magnitudo (panjang) dari keadaan awal $|\psi\rangle$ dikuadratkan adalah $\langle\psi|\psi\rangle$. Jika keadaan awal ditransformasikan oleh operator uniter menjadi $|\psi'\rangle = \hat{U}|\psi\rangle$, maka *bra*-nya menjadi $\langle\psi'| = \langle\psi|\hat{U}^\dagger$.
Kuadrat magnitudo keadaan baru adalah:
$$\langle\psi'|\psi'\rangle = (\langle\psi|\hat{U}^\dagger) (\hat{U}|\psi\rangle) = \langle\psi| (\hat{U}^\dagger\hat{U}) |\psi\rangle.$$
Berdasarkan definisi operator uniter, $\hat{U}^\dagger\hat{U} = \hat{1}$, sehingga:
$$\langle\psi'|\psi'\rangle = \langle\psi|\hat{1}|\psi\rangle = \langle\psi|\psi\rangle. \quad \blacksquare$$

**(ii)** Operator proyeksi didefinisikan sebagai $\hat{P}_\psi = |\psi\rangle\langle\psi|$. Jika dikuadratkan:
$$\hat{P}_\psi^2 = (|\psi\rangle\langle\psi|) (|\psi\rangle\langle\psi|) = |\psi\rangle (\langle\psi|\psi\rangle) \langle\psi|.$$
Karena vektor keadaan $|\psi\rangle$ ter-normalisasi ($\langle\psi|\psi\rangle = 1$), maka:
$$\hat{P}_\psi^2 = |\psi\rangle (1) \langle\psi| = |\psi\rangle\langle\psi| = \hat{P}_\psi. \quad \blacksquare$$

In [21]:
# Verifikasi Komputasi Soal 1

# (i) Verifikasi magnitudo pada 5 vektor acak dengan operator uniter acak / rotasi
print("Verifikasi Soal 1(i): Pelestarian Magnitudo oleh Operator Uniter")
U_test = R_p(53.13) # Operator uniter rotasi
ket_rnd = clean_array(np.array([[3 + 4j], [1 - 2j]]))
ket_rnd = ket_rnd / norm(ket_rnd) # dinormalisasi

mag_awal = inner_product(ket_rnd, ket_rnd).real
ket_trans = U_test @ ket_rnd
mag_akhir = inner_product(ket_trans, ket_trans).real

print(f"  <psi|psi> awal   = {mag_awal:.6f}")
print(f"  <psi'|psi'> akhir = {mag_akhir:.6f}")
print(f"  Terbukti sama? {np.isclose(mag_awal, mag_akhir)}")
print()

# (ii) Verifikasi idempoten P^2 == P pada proyektor sembarang
print("Verifikasi Soal 1(ii): Idempoten P_psi^2 == P_psi")
P_rnd = outer_product(ket_rnd, ket_rnd)
print("Apakah P_rnd @ P_rnd == P_rnd?", np.allclose(P_rnd @ P_rnd, P_rnd))

Verifikasi Soal 1(i): Pelestarian Magnitudo oleh Operator Uniter
  <psi|psi> awal   = 1.000000
  <psi'|psi'> akhir = 1.000000
  Terbukti sama? True

Verifikasi Soal 1(ii): Idempoten P_psi^2 == P_psi
Apakah P_rnd @ P_rnd == P_rnd? True


## Soal 2
**Dengan menggunakan representasi matriks dari operator rotasi polarisasi $\hat{R}_p(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$, buktikan:**
**(i) operator tersebut uniter,**
**(ii) $\hat{R}_p^\dagger(\theta) = \hat{R}_p(-\theta)$,**
**(iii) $\hat{R}_p(\theta)|\phi\rangle = |\phi+\theta\rangle$, dengan $|\phi\rangle = \begin{pmatrix} \cos\phi \\ \sin\phi \end{pmatrix}$.**

### Pembuktian Analitik:
**(i)** Matriks *adjoint* diperoleh dengan transpos konjugat (karena semua elemen riil, cukup ditranspos):
$$\mathbf{R}_p^\dagger(\theta) = \begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix}.$$
Kalikan keduanya:
$$\mathbf{R}_p^\dagger(\theta) \mathbf{R}_p(\theta) = \begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} = \begin{pmatrix} \cos^2\theta + \sin^2\theta & -\cos\theta\sin\theta + \sin\theta\cos\theta \\ -\sin\theta\cos\theta + \cos\theta\sin\theta & \sin^2\theta + \cos^2\theta \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = \mathbf{I}. \quad \blacksquare$$

**(ii)** Substitusi sudut $-\theta$ ke dalam matriks rotasi (mengingat $\cos(-\theta) = \cos\theta$ dan $\sin(-\theta) = -\sin\theta$):
$$\mathbf{R}_p(-\theta) = \begin{pmatrix} \cos(-\theta) & -\sin(-\theta) \\ \sin(-\theta) & \cos(-\theta) \end{pmatrix} = \begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix}.$$
Hasil ini persis sama dengan matriks $\mathbf{R}_p^\dagger(\theta)$ yang diperoleh pada poin (i). $\blacksquare$

**(iii)** Perkalian matriks dengan vektor kolom $|\phi\rangle$:
$$\mathbf{R}_p(\theta)|\phi\rangle = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} \cos\phi \\ \sin\phi \end{pmatrix} = \begin{pmatrix} \cos\theta\cos\phi - \sin\theta\sin\phi \\ \sin\theta\cos\phi + \cos\theta\sin\phi \end{pmatrix}.$$
Menggunakan rumus trigonometri penjumlahan sudut, baris atas adalah $\cos(\theta+\phi)$ dan baris bawah adalah $\sin(\theta+\phi)$:
$$\mathbf{R}_p(\theta)|\phi\rangle = \begin{pmatrix} \cos(\phi+\theta) \\ \sin(\phi+\theta) \end{pmatrix} = |\phi+\theta\rangle. \quad \blacksquare$$

In [22]:
# Verifikasi Komputasi Soal 2 dengan SymPy

theta, phi = sp.symbols('theta phi', real=True)
R = sp.Matrix([[sp.cos(theta), -sp.sin(theta)],
               [sp.sin(theta),  sp.cos(theta)]])

R_dagger = R.H  # Conjugate transpose di SymPy

print("Verifikasi Soal 2(i): R^dagger * R == I")
I_check = sp.simplify(R_dagger * R)
sp.pprint(I_check)
print()

print("Verifikasi Soal 2(ii): R^dagger == R(-theta)")
R_min_theta = sp.Matrix([[sp.cos(-theta), -sp.sin(-theta)],
                         [sp.sin(-theta),  sp.cos(-theta)]])
print("Apakah R_dagger == R(-theta)?", sp.simplify(R_dagger - R_min_theta) == sp.zeros(2, 2))
print()

print("Verifikasi Soal 2(iii): R(theta)*|phi> == |phi+theta>")
ket_phi = sp.Matrix([[sp.cos(phi)], [sp.sin(phi)]])
res_rot = sp.simplify(R * ket_phi)
sp.pprint(res_rot)

Verifikasi Soal 2(i): R^dagger * R == I
⎡1  0⎤
⎢    ⎥
⎣0  1⎦

Verifikasi Soal 2(ii): R^dagger == R(-theta)
Apakah R_dagger == R(-theta)? True

Verifikasi Soal 2(iii): R(theta)*|phi> == |phi+theta>
⎡cos(φ + θ)⎤
⎢          ⎥
⎣sin(φ + θ)⎦


## Soal 3
**Hitunglah nilai-nilai *eigen* dan keadaan-keadaan *eigen* dari operator:**
**(i) $\hat{P}_{+45} = |+45\rangle\langle +45|$,**
**(ii) $\hat{R}_p(\theta)$.**

### Pembuktian Analitik:
**(i) Untuk $\hat{P}_{+45}$:**
Matriks dalam basis $HV$ adalah $\mathbf{P}_{+45} = \begin{pmatrix} 1/2 & 1/2 \\ 1/2 & 1/2 \end{pmatrix}$. Persamaan karakteristik $\det(\mathbf{P}_{+45} - \lambda \mathbf{I}) = 0$:
$$\left(\frac{1}{2} - \lambda\right)^2 - \frac{1}{4} = 0 \implies \lambda^2 - \lambda = 0 \implies \lambda_1 = 1, \quad \lambda_2 = 0.$$
- Untuk $\lambda_1 = 1$: $\frac{1}{2}x + \frac{1}{2}y = x \implies x = y$. Vektor ter-normalisasi adalah $\frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} = |+45\rangle$.
- Untuk $\lambda_2 = 0$: $\frac{1}{2}x + \frac{1}{2}y = 0 \implies x = -y$. Vektor ter-normalisasi adalah $\frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -1 \end{pmatrix} = |-45\rangle$.

**(ii) Untuk $\hat{R}_p(\theta)$:**
Persamaan karakteristik $\det(\mathbf{R}_p(\theta) - \lambda \mathbf{I}) = 0$:
$$(\cos\theta - \lambda)^2 + \sin^2\theta = 0 \implies \lambda^2 - 2\lambda\cos\theta + 1 = 0.$$
Dengan rumus kuadratik (atau rumus Euler): $\lambda = \cos\theta \pm i\sin\theta = e^{\pm i\theta}$.
- Untuk $\lambda_1 = e^{i\theta}$: $(\cos\theta - e^{i\theta})x - \sin\theta y = 0 \implies (-i\sin\theta)x = \sin\theta y \implies y = -ix$. Vektor ter-normalisasi adalah $\frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -i \end{pmatrix} = |R\rangle$ (Polarisasi melingkar kanan).
- Untuk $\lambda_2 = e^{-i\theta}$: dengan cara serupa diperoleh $y = ix$. Vektor ter-normalisasi adalah $\frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ i \end{pmatrix} = |L\rangle$ (Polarisasi melingkar kiri).

In [23]:
# Verifikasi Komputasi Soal 3

print("--- Soal 3(i): Nilai & Vektor Eigen P_+45 ---")
evals_P, evecs_P = np.linalg.eig(P_plus45)
for idx in range(2):
    val = evals_P[idx]
    vec = evecs_P[:, idx:idx+1]
    print(f"Nilai eigen: {val:.4f}")
    print_ket(f"Vektor eigen (bersesuaian)", vec)

print("--- Soal 3(ii): Nilai & Vektor Eigen R_p(theta) secara Simbolik ---")
eigen_data = R.eigenvects()
for val, mult, vecs in eigen_data:
    print("Nilai eigen:")
    sp.pprint(sp.simplify(val))
    print("Vektor eigen:")
    sp.pprint(sp.simplify(vecs[0]))
    print()

--- Soal 3(i): Nilai & Vektor Eigen P_+45 ---
Nilai eigen: 1.0000+0.0000j
|Vektor eigen (bersesuaian)> =
[[0.7071]
 [0.7071]]

Nilai eigen: -0.0000+0.0000j
|Vektor eigen (bersesuaian)> =
[[ 0.7071]
 [-0.7071]]

--- Soal 3(ii): Nilai & Vektor Eigen R_p(theta) secara Simbolik ---
Nilai eigen:


cos(θ) - ⅈ⋅│sin(θ)│
Vektor eigen:
⎡-ⅈ⋅sin(θ) ⎤
⎢──────────⎥
⎢ │sin(θ)│ ⎥
⎢          ⎥
⎣    1     ⎦

Nilai eigen:
cos(θ) + ⅈ⋅│sin(θ)│
Vektor eigen:
⎡ⅈ⋅sin(θ)⎤
⎢────────⎥
⎢│sin(θ)│⎥
⎢        ⎥
⎣   1    ⎦



## Soal 4
**Hitunglah $e^{\hat{O}} |\lambda\rangle$ dengan mengasumsikan $|\lambda\rangle$ merupakan suatu keadaan *eigen* dari operator $\hat{O}$ dengan nilai *eigen* $\lambda$.**

### Pembuktian Analitik:
Berdasarkan definisi deret Taylor untuk fungsi eksponensial operator (Persamaan 10):
$$e^{\hat{O}} |\lambda\rangle = \left( \sum_{n=0}^\infty \frac{1}{n!} \hat{O}^n \right) |\lambda\rangle = \sum_{n=0}^\infty \frac{1}{n!} \bigl(\hat{O}^n |\lambda\rangle\bigr).$$
Karena $|\lambda\rangle$ adalah keadaan *eigen*, aplikasi operator $\hat{O}$ sebanyak satu kali menghasilkan $\lambda|\lambda\rangle$. Aplikasi sebanyak $n$ kali menghasilkan:
$$\hat{O}^n |\lambda\rangle = \hat{O}^{n-1}(\hat{O}|\lambda\rangle) = \hat{O}^{n-1}(\lambda|\lambda\rangle) = \lambda^n |\lambda\rangle.$$
Substitusi kembali ke dalam deret sumasi:
$$e^{\hat{O}} |\lambda\rangle = \sum_{n=0}^\infty \frac{1}{n!} \lambda^n |\lambda\rangle = \left( \sum_{n=0}^\infty \frac{\lambda^n}{n!} \right) |\lambda\rangle.$$
Sumasi di dalam kurung tiada lain adalah ekspansi deret Taylor standar untuk skalar $e^\lambda$. Dengan demikian:
$$e^{\hat{O}} |\lambda\rangle = e^\lambda |\lambda\rangle. \quad \blacksquare$$

In [24]:
# Verifikasi Komputasi Soal 4

print("Verifikasi Soal 4: e^O |lam> == e^lam |lam>")

# Kita ambil operator O = R_p(30 deg), yang memiliki vektor eigen |R> dengan nilai eigen e^(i * 30 deg)
theta_rad = np.deg2rad(30)
O_mat = R_p(30)
lam_val = np.exp(1j * theta_rad)
ket_eigen = ket_R

# Hitung e^O secara matriks menggunakan expm dari SciPy
exp_O_mat = expm(O_mat)

# Aksi e^O pada vektor eigen |lam>
lhs = exp_O_mat @ ket_eigen

# Perkalian skalar e^lam pada vektor eigen |lam>
rhs = np.exp(lam_val) * ket_eigen

print_ket("LHS: expm(O) |R>", lhs)
print_ket("RHS: exp(lam) |R>", rhs)
print("Apakah e^O |lam> == e^lam |lam>?", np.allclose(lhs, rhs))

Verifikasi Soal 4: e^O |lam> == e^lam |lam>
|LHS: expm(O) |R>> =
[[1.4753+0.806j ]
 [0.806 -1.4753j]]

|RHS: exp(lam) |R>> =
[[1.4753+0.806j ]
 [0.806 -1.4753j]]

Apakah e^O |lam> == e^lam |lam>? True
